In [4]:
# Verify GPU
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

# Uninstall and reinstall libraries to fix bitsandbytes and transformers issues
!pip uninstall -y transformers datasets peft torch pandas bitsandbytes accelerate
!pip install transformers==4.45.2 datasets==3.0.1 peft==0.13.2 torch==2.4.1 pandas==2.2.3 bitsandbytes==0.44.1 accelerate==1.0.1 --no-cache-dir

# Verify transformers version
import transformers
print("Transformers version:", transformers.__version__)

CUDA available: True
GPU name: Tesla T4
Found existing installation: transformers 4.45.2
Uninstalling transformers-4.45.2:
  Successfully uninstalled transformers-4.45.2
Found existing installation: datasets 3.0.1
Uninstalling datasets-3.0.1:
  Successfully uninstalled datasets-3.0.1
Found existing installation: peft 0.13.2
Uninstalling peft-0.13.2:
  Successfully uninstalled peft-0.13.2
Found existing installation: torch 2.4.1
Uninstalling torch-2.4.1:
  Successfully uninstalled torch-2.4.1
Found existing installation: pandas 2.2.3
Uninstalling pandas-2.2.3:
  Successfully uninstalled pandas-2.2.3
Found existing installation: accelerate 1.9.0
Uninstalling accelerate-1.9.0:
  Successfully uninstalled accelerate-1.9.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 137.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 167.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 183.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Transformers version: 4.45.2


In [1]:
import pandas as pd

passages_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")
test_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")
print("Test DataFrame size:", len(test_df))
print("Test columns:", test_df.columns)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Test DataFrame size: 4719
Test columns: Index(['question', 'answer', 'relevant_passage_ids'], dtype='object')


In [2]:
from datasets import Dataset

def format_data(row):
    return {"text": f"Question: {row['question']}\nAnswer: {row['answer']}"}

data = test_df.apply(format_data, axis=1).tolist()
dataset = Dataset.from_list(data)
dataset = dataset.train_test_split(test_size=0.2)
train_dataset = dataset["train"]
val_dataset = dataset["test"]
print("Train dataset size:", len(train_dataset))

Train dataset size: 3775


In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16
)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# Enable gradients for LoRA parameters
model.train()  # Set model to training mode
for name, param in model.named_parameters():
    if "lora" in name.lower():
        param.requires_grad = True

# Verify trainable parameters
model.print_trainable_parameters()
print("Trainable parameters with requires_grad=True:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044
Trainable parameters with requires_grad=True:
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight
base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight
base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight
base_model.model.model.layers.1.self_attn.v_proj.lora_A.default.weight
base_model.model.model.layers.1.self_attn.v_proj.lora_B.default.weight
base_model.model.model.layers.2.self_attn.q_proj.lora_A.default.weight
base_model.model.model.layers.2.self_attn.q_proj.lora_B.default.weight
base_model.model.model.layers.2.self_attn.v_proj.lora_A.default.weight
base_model.model.model.layers.2.self_attn.v_proj.lora_B.default.weight
base_model.model.model.

In [4]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

# Load tokenized datasets if saved, or tokenize anew
try:
    from datasets import load_from_disk
    tokenized_train = load_from_disk("./tokenized_train")
    tokenized_val = load_from_disk("./tokenized_val")
except:
    tokenized_train = train_dataset.map(tokenize_function, batched=True)
    tokenized_val = val_dataset.map(tokenize_function, batched=True)
    tokenized_train = tokenized_train.remove_columns(["text"])
    tokenized_val = tokenized_val.remove_columns(["text"])
    tokenized_train.set_format("torch")
    tokenized_val.set_format("torch")
    tokenized_train.save_to_disk("./tokenized_train")
    tokenized_val.save_to_disk("./tokenized_val")
print("Tokenized train sample:", tokenized_train[0])

Map:   0%|          | 0/3775 [00:00<?, ? examples/s]

Map:   0%|          | 0/944 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3775 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/944 [00:00<?, ? examples/s]

Tokenized train sample: {'input_ids': tensor([    1,   894, 29901,  1317,   727,   263,  9443,  1546,  4707,   312,
          262,   322,   364, 10094,   397,   457,   337,  1547,   943, 29973,
           13, 22550, 29901,  3869, 29892,  4707,   312,   262,  7868, 29879,
          304,   364, 10094,   397,   457,   337,  1547,   943,  2629,   278,
          432,   651,   284,   269,  5666,   459,  3333, 13076,  3240, 12906,
          398,   310, 15835,   398,  6507, 10340, 29892,   322, 12891, 14741,
          408,   385,  5039,  1061,   310, 15586, 29934, 18196,   472,  4482,
        19703,   979,   518, 26270, 29898, 29906, 28135,  1402,   322,   408,
          385,   297,  6335,  2105,   472,  1880, 19703,   979,   518, 26270,
        29898, 29906, 28135,  1822,     2,     2,     2,     2,     2,     2,
            2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
            2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
            2,     2,     

In [10]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./results",
    run_name="bioasq_tinyllama_finetune",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    fp16=True,
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    max_grad_norm=0.5,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
)

trainer.train()
trainer.save_model("./fine_tuned_model")
tokenizer.save_pretrained("./fine_tuned_model")

Epoch,Training Loss,Validation Loss
1,1.725300,1.784457
2,1.698600,1.776631
3,1.612100,1.780606


('./fine_tuned_model/tokenizer_config.json',
 './fine_tuned_model/special_tokens_map.json',
 './fine_tuned_model/tokenizer.model',
 './fine_tuned_model/added_tokens.json',
 './fine_tuned_model/tokenizer.json')

In [11]:
model.eval()
test_input = "Question: What is the main cause of lung cancer?\nAnswer:"
inputs = tokenizer(test_input, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


Question: What is the main cause of lung cancer?
Answer: Lung cancer is the leading cause of cancer deaths worldwide. The most common cause of lung cancer is smoking, followed by exposure to asbestos, radon, and occupational exposure to silica. The second most common cause of lung cancer is chronic obstructive pulmonary disease (COPD). The third most common cause of lung cancer is alcoholic lung disease. The fourth most common cause of lung cancer is exposure to asbestos. The


In [12]:
def chatbot():
    print("BioASQ Chatbot: Type 'quit' to exit.")
    while True:
        user_input = input("You: ")
        if user_input.lower() == "quit":
            break
        prompt = f"Question: {user_input}\nAnswer:"
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=100, do_sample=True)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print("Bot:", response.split("Answer:")[1].strip())

chatbot()

BioASQ Chatbot: Type 'quit' to exit.
You: What is the main cause of lung cancer?
Bot: The main cause of lung cancer is smoking, while the second most important risk factor is cigarette smoking alone (approximately 40% of lung cancer cases) . Cigarette smoking, particularly its high-nicotine content , produces oxidative damage, with consequent activation of inflammatory proteins like TNFα, TNFRSF1A, and MDC. These mediators induce fibroblast and epithelial
You: quit


## Next Steps
### 1. Add RAG (Retrieval-Augmented Generation) using FAISS
### 2. Try QLoRA for lower memory footprint
### 3. Wrap chatbot with Gradio or Streamlit
### 4. Test with LLaMA2-7B if more powerful GPU is available